# Kvasir-VQA x1 — Image-only baseline (best-practice)

This notebook builds a reproducible image-only baseline with frozen ViT embeddings + Logistic Regression.
It includes consistent label sets, optional image-disjoint splitting, and simple baselines for context.


In [1]:
# Performance config
import os

CPU_THREADS = min(32, os.cpu_count() or 32)
os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)
os.environ["NUMEXPR_NUM_THREADS"] = str(CPU_THREADS)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(CPU_THREADS)
os.environ["TOKENIZERS_PARALLELISM"] = "true"

REQUIRE_CUDA = False  # set True to enforce GPU

print(f"CPU threads set to: {CPU_THREADS}")

try:
    import torch
except Exception as e:
    torch = None
    if REQUIRE_CUDA:
        raise RuntimeError("CUDA required but torch is not available.") from e

if torch is not None:
    torch.set_num_threads(CPU_THREADS)
    torch.set_num_interop_threads(min(4, CPU_THREADS))
    if REQUIRE_CUDA and not torch.cuda.is_available():
        raise RuntimeError("CUDA required but not available.")
    if torch.cuda.is_available():
        torch.backends.cuda.matmul.allow_tf32 = True
        print("CUDA device:", torch.cuda.get_device_name(0))
    else:
        print("CUDA not available, using CPU.")


CPU threads set to: 12
CUDA device: NVIDIA GeForce RTX 5070 Ti


In [2]:
# Imports
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, balanced_accuracy_score, top_k_accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoImageProcessor, ViTModel


2026-02-01 17:21:07.892102: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-02-01 17:21:07.892141: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-02-01 17:21:07.893520: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-02-01 17:21:07.900325: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-01 17:21:08.629773: W tensorflow/compiler/tf2

In [3]:
# Config
MODEL_NAME = "google/vit-base-patch16-224-in21k"
BATCH_SIZE = 16
NUM_WORKERS = 0
TOP_K = 200
SEED = 42
VAL_FRACTION = 0.1  # used if no validation split exists
USE_IMAGE_DISJOINT_SPLIT = False  # set True to avoid image leakage for image-only baselines
CLASS_WEIGHT = None  # set "balanced" to counter class imbalance
REUSE_LABELS = True  # reuse saved label list for comparability

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(SEED)
np.random.seed(SEED)
if torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

print("Device:", DEVICE)


Device: cuda


In [4]:
# Paths & dataset root

def find_kvasir_x1_root() -> Path:
    import os
    env_root = os.environ.get("KVASIR_VQA_X1_ROOT")
    if env_root:
        p = Path(env_root).expanduser().resolve()
        if (p / "0_dataset_prep").exists():
            return p
        raise RuntimeError(f"KVASIR_VQA_X1_ROOT set but missing 0_dataset_prep: {p}")

    if "__file__" in globals():
        p = Path(__file__).resolve()
        root = p.parents[2]
        if root.name == "Kvasir_VQA_x1" and (root / "0_dataset_prep").exists():
            return root

    cwd = Path.cwd().resolve()
    for p in [cwd] + list(cwd.parents):
        if p.name == "Kvasir_VQA_x1" and (p / "0_dataset_prep").exists():
            return p

    raise RuntimeError( "Could not locate Kvasir_VQA_x1 dataset root." "Run this notebook from within the Kvasir_VQA_x1 folder,"
        "or set KVASIR_VQA_X1_ROOT."
    )

DATA_ROOT = find_kvasir_x1_root()
META_CSV = DATA_ROOT / "0_dataset_prep" / "out" / "metadata" / "metadata_enriched.csv"
OUT_DIR = DATA_ROOT / "2_modeling" / "02_image_only" / "out" / "baseline_best"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data root:", DATA_ROOT)
print("Metadata:", META_CSV)
print("Out dir:", OUT_DIR)


Data root: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1
Metadata: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/0_dataset_prep/out/metadata/metadata_enriched.csv
Out dir: /home/aristotle/Desktop/rag-vqa-medical/Prototyping_reformat/DatasetAnalysis/Kvasir_VQA_x1/2_modeling/02_image_only/out/baseline_best


In [5]:
# Load metadata
meta = pd.read_csv(META_CSV)

required_cols = {"img_id", "image_path", "answer"}
missing = required_cols - set(meta.columns)
if missing:
    raise RuntimeError(f"Missing required columns: {missing}")

# Resolve image paths relative to dataset root if needed
images_base = DATA_ROOT / "0_dataset_prep"
meta["image_path"] = meta["image_path"].apply(
    lambda p: str((images_base / p).resolve()) if not Path(p).is_absolute() else p
)

meta["answer_norm"] = meta["answer"].fillna("").astype(str).str.lower().str.strip()

print("Rows:", len(meta))


Rows: 159549


In [6]:
# Build splits

if USE_IMAGE_DISJOINT_SPLIT:
    # Split by unique images to avoid leakage for image-only baselines
    unique_imgs = meta[["img_id"]].drop_duplicates()
    img_ids = unique_imgs["img_id"].values
    train_ids, test_ids = train_test_split(
        img_ids, test_size=0.2, random_state=SEED, shuffle=True
    )
    train_ids, val_ids = train_test_split(
        train_ids, test_size=VAL_FRACTION, random_state=SEED, shuffle=True
    )
    train_df = meta[meta["img_id"].isin(train_ids)].reset_index(drop=True)
    val_df = meta[meta["img_id"].isin(val_ids)].reset_index(drop=True)
    test_df = meta[meta["img_id"].isin(test_ids)].reset_index(drop=True)
    split_source = "image-disjoint"
else:
    if "split" not in meta.columns:
        raise RuntimeError("Missing 'split' column. Run dataset prep split step first.")

    train_df = meta[meta["split"] == "train"].reset_index(drop=True)
    test_df = meta[meta["split"] == "test"].reset_index(drop=True)
    val_df = meta[meta["split"] == "validation"].reset_index(drop=True)

    if len(val_df) == 0:
        # Create a stratified validation split from train if none exists
        y = train_df["answer_norm"]
        class_counts = y.value_counts()
        rare_classes = class_counts[class_counts < 2].index

        if len(rare_classes) > 0:
            # Keep rare classes in train only to satisfy stratification constraints
            rare_mask = y.isin(rare_classes)
            train_common = train_df[~rare_mask].reset_index(drop=True)
            train_rare = train_df[rare_mask].reset_index(drop=True)

            if train_common["answer_norm"].nunique() < 2:
                # Fallback to non-stratified split if too few classes remain
                idx_train, idx_val = train_test_split(
                    train_df.index, test_size=VAL_FRACTION, random_state=SEED, shuffle=True
                )
                val_df = train_df.loc[idx_val].reset_index(drop=True)
                train_df = train_df.loc[idx_train].reset_index(drop=True)
                split_source = "train/test + random val (fallback; insufficient class counts)"
            else:
                n_samples = len(train_common)
                n_test = int(np.ceil(n_samples * VAL_FRACTION))
                n_train = n_samples - n_test
                n_classes = train_common["answer_norm"].nunique()

                if n_test < n_classes or n_train < n_classes:
                    # Fallback to random split when stratification is impossible
                    idx_train, idx_val = train_test_split(
                        train_common.index, test_size=VAL_FRACTION, random_state=SEED, shuffle=True
                    )
                    val_df = train_common.loc[idx_val].reset_index(drop=True)
                    train_df = train_common.loc[idx_train].reset_index(drop=True)
                    train_df = pd.concat([train_df, train_rare], ignore_index=True)
                    split_source = "train/test + random val (fallback; too many classes)"
                else:
                    splitter = StratifiedShuffleSplit(
                        n_splits=1, test_size=VAL_FRACTION, random_state=SEED
                    )
                    idx_train, idx_val = next(
                        splitter.split(train_common, train_common["answer_norm"])
                    )
                    val_df = train_common.iloc[idx_val].reset_index(drop=True)
                    train_df = train_common.iloc[idx_train].reset_index(drop=True)
                    # Add rare classes back into train only
                    train_df = pd.concat([train_df, train_rare], ignore_index=True)
                    split_source = "train/test + stratified val from common; rare kept in train"
        else:
            n_samples = len(train_df)
            n_test = int(np.ceil(n_samples * VAL_FRACTION))
            n_train = n_samples - n_test
            n_classes = train_df["answer_norm"].nunique()

            if n_test < n_classes or n_train < n_classes:
                idx_train, idx_val = train_test_split(
                    train_df.index, test_size=VAL_FRACTION, random_state=SEED, shuffle=True
                )
                val_df = train_df.loc[idx_val].reset_index(drop=True)
                train_df = train_df.loc[idx_train].reset_index(drop=True)
                split_source = "train/test + random val (fallback; too many classes)"
            else:
                splitter = StratifiedShuffleSplit(
                    n_splits=1, test_size=VAL_FRACTION, random_state=SEED
                )
                idx_train, idx_val = next(
                    splitter.split(train_df, train_df["answer_norm"])
                )
                val_df = train_df.iloc[idx_val].reset_index(drop=True)
                train_df = train_df.iloc[idx_train].reset_index(drop=True)
                split_source = "train/test + stratified val from train"
    else:
        split_source = "predefined train/val/test"

print("Split source:", split_source)
print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})


Split source: train/test + random val (fallback; too many classes)
{'train': 135427, 'val': 8167, 'test': 15955}


In [7]:
# Top-K answers (labels)
labels_path = OUT_DIR / f"labels_topk_{TOP_K}.json"

if REUSE_LABELS and labels_path.exists():
    with open(labels_path, "r") as f:
        LABELS = json.load(f)
    print("Loaded labels:", len(LABELS))
else:
    answer_counts = train_df["answer_norm"].value_counts()
    LABELS = answer_counts.head(TOP_K).index.tolist()
    with open(labels_path, "w") as f:
        json.dump(LABELS, f, indent=2)
    print("Saved labels:", len(LABELS))

# Filter to Top-K
train_k = train_df[train_df["answer_norm"].isin(LABELS)].reset_index(drop=True)
val_k = val_df[val_df["answer_norm"].isin(LABELS)].reset_index(drop=True)
test_k = test_df[test_df["answer_norm"].isin(LABELS)].reset_index(drop=True)

print({"train": len(train_k), "val": len(val_k), "test": len(test_k)})
print("Classes in train:", train_k["answer_norm"].nunique())


Loaded labels: 200
{'train': 34541, 'val': 3877, 'test': 4251}
Classes in train: 200


In [8]:
# Simple baselines

def eval_baseline(y_true, y_pred, split_name, label_list):
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", labels=label_list, zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", labels=label_list, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }
    print(f"{split_name} baseline", metrics)

majority_label = train_k["answer_norm"].value_counts().idxmax()

for name, df in [("train", train_k), ("val", val_k), ("test", test_k)]:
    if len(df) == 0:
        continue
    y = df["answer_norm"].values
    y_pred = np.full_like(y, majority_label)
    eval_baseline(y, y_pred, name, LABELS)


train baseline {'accuracy': 0.045105816276309316, 'macro_f1': 0.00043159090279509127, 'weighted_f1': 0.0038934519936003717, 'balanced_accuracy': 0.005}
val baseline {'accuracy': 0.04152695383028115, 'macro_f1': 0.0003987122337790985, 'weighted_f1': 0.0033114609047425773, 'balanced_accuracy': 0.005}
test baseline {'accuracy': 0.04328393319219007, 'macro_f1': 0.00041488162344983085, 'weighted_f1': 0.0035915416944139675, 'balanced_accuracy': 0.005}


In [9]:
# Image embedding extraction
class ImageDS(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row["img_id"]
        img = Image.open(row["image_path"]).convert("RGB")
        return img_id, img


def collate_fn(batch):
    ids = [b[0] for b in batch]
    images = [b[1] for b in batch]
    inputs = processor(images=images, return_tensors="pt")
    return ids, inputs["pixel_values"]


def compute_embeddings(unique_df):
    ds = ImageDS(unique_df)
    dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, collate_fn=collate_fn)
    emb_map = {}
    with torch.no_grad():
        for ids, pixels in tqdm(dl, desc="ViT embed"):
            pixels = pixels.to(DEVICE)
            out = vit(pixels).last_hidden_state[:, 0, :]
            for i, img_id in enumerate(ids):
                emb_map[img_id] = out[i].cpu().numpy()
    return emb_map


In [10]:
# Compute or load cached embeddings
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
vit = ViTModel.from_pretrained(MODEL_NAME).to(DEVICE)
vit.eval()

model_tag = MODEL_NAME.replace("/", "_")
EMB_PATH = OUT_DIR / f"image_embeddings_{model_tag}.npz"

unique_imgs = pd.concat([train_k, val_k, test_k])[["img_id", "image_path"]].drop_duplicates()

if EMB_PATH.exists():
    data = np.load(EMB_PATH, allow_pickle=True)
    img_ids = data["img_ids"].tolist()
    embeddings = data["embeddings"]
    emb_map = {img_id: embeddings[i] for i, img_id in enumerate(img_ids)}
    print("Loaded cached embeddings:", len(emb_map))
else:
    emb_map = compute_embeddings(unique_imgs)
    img_ids = list(emb_map.keys())
    embeddings = np.stack([emb_map[i] for i in img_ids])
    np.savez(EMB_PATH, img_ids=np.array(img_ids), embeddings=embeddings)
    print("Saved embeddings:", EMB_PATH)


Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


Loaded cached embeddings: 6329


In [11]:
# Build feature matrices

def build_X(df):
    return np.stack([emb_map[i] for i in df["img_id"].tolist()])

X_train = build_X(train_k)
X_val = build_X(val_k) if len(val_k) else None
X_test = build_X(test_k) if len(test_k) else None

y_train = train_k["answer_norm"].values
y_val = val_k["answer_norm"].values if len(val_k) else None
y_test = test_k["answer_norm"].values if len(test_k) else None

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
if X_val is not None:
    X_val = scaler.transform(X_val)
if X_test is not None:
    X_test = scaler.transform(X_test)


In [12]:
# Train classifier
clf = LogisticRegression(max_iter=1000, n_jobs=-1, class_weight=CLASS_WEIGHT)
clf.fit(X_train, y_train)


def eval_split(X, y_true, split_name):
    y_pred = clf.predict(X)
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", labels=LABELS, zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", labels=LABELS, zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
    }
    # Top-5 accuracy if probabilities available
    if hasattr(clf, "predict_proba"):
        y_proba = clf.predict_proba(X)
        # top_k_accuracy_score expects labels ordered to match y_proba columns
        labels_topk = getattr(clf, "classes_", LABELS)
        metrics["top5_accuracy"] = float(
            top_k_accuracy_score(y_true, y_proba, k=5, labels=labels_topk)
        )

    report = classification_report(y_true, y_pred, labels=LABELS, output_dict=True, zero_division=0)
    pred_df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred})
    pred_df.to_csv(OUT_DIR / f"pred_{split_name}.csv", index=False)
    with open(OUT_DIR / f"metrics_{split_name}.json", "w") as f:
        json.dump({"metrics": metrics, "report": report}, f, indent=2)

    print(split_name, metrics)


# Evaluate
if len(train_k):
    eval_split(X_train, y_train, "train")
if X_val is not None:
    eval_split(X_val, y_val, "val")
if X_test is not None:
    eval_split(X_test, y_test, "test")


train {'accuracy': 0.1666714918502649, 'macro_f1': 0.1530163932422235, 'weighted_f1': 0.1582593707089302, 'balanced_accuracy': 0.15867522504278128, 'top5_accuracy': 0.5794273472105613}
val {'accuracy': 0.0196027856590147, 'macro_f1': 0.008339653005038257, 'weighted_f1': 0.019361135954765872, 'balanced_accuracy': 0.008351928063991706, 'top5_accuracy': 0.08847046685581635}
test {'accuracy': 0.020465772759350742, 'macro_f1': 0.006788312050077041, 'weighted_f1': 0.01928072930913461, 'balanced_accuracy': 0.006901538291683179, 'top5_accuracy': 0.09221359680075276}


## Notes
- If `USE_IMAGE_DISJOINT_SPLIT=True`, the baseline avoids image leakage for image-only models, but the split no longer matches the dataset's original QA split.
- `labels_topk_{TOP_K}.json` is saved to keep label sets consistent across techniques.
